# 🎭 AI Companion: Universal Roleplay Bridge (Threaded)

This notebook acts as a remote 'brain' for your AI Companion. It allows you to run high-end models on Google's T4 GPUs or Kaggle and tunnel the response back to your local machine via Ngrok.

### 🛠️ Setup Instructions:
1. **Environment**: In the cell below, change `ENV_TYPE` to match your environment first.
2. **GPU Acceleration**: Go to `Runtime` > `Change runtime type` and ensure **T4 GPU** is selected.
3. **Secrets (IMPORTANT)**: 
   - **Colab**: Click the **Key icon** (Secrets) on the left sidebar.
   - **Kaggle**: Go to **Add-ons** > **Secrets**.
   - Add `HF_TOKEN` with your [HuggingFace Token](https://huggingface.co/settings/tokens).
   - Add `NGROK_TOKEN` with your [Ngrok Authtoken](https://dashboard.ngrok.com/get-started/your-authtoken).
   - Enable access for the notebook.
4. **Run All**: Press `Ctrl + F9` or go to `Runtime` > `Run all`.

### 🔗 Connecting to the Local App:
1. Wait for the final cell to display the **🚀 BRIDGE ONLINE!** message.
2. Copy the **URL** (it will look like `https://xxxx-xx-xx-xx.ngrok-free.app`).
3. Open your local `settings.json` and paste the URL into `remote_llm_url`.
4. Restart your local `main.py` script.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Simply change this variable to "colab" or "kaggle" based on your environment. The code will attempt to load secrets accordingly.
ENV_TYPE = "kaggle"

# VS Code Colab Runtime Fix: Explicitly check for .env in mounted paths if local load fails
if not os.environ.get("HF_TOKEN"):
    # Try common VS Code Colab mount points or home directory
    for path in ["/content/.env", os.path.expanduser("~/.env")]:
        if os.path.exists(path):
            load_dotenv(path)
            break

def get_secret(name):
    # 1. Check environment variables (Local/.env)
    val = os.environ.get(name)
    if val: return val.strip()
    # 2. Check Colab Secrets
    if ENV_TYPE == "colab":
        try:
            from google.colab import userdata
            s = userdata.get(name)
            return s.strip() if s else None
        except:
            print("❌ [ERROR] Could not access Colab secrets. Make sure to set them up correctly.")
    # 3. Check Kaggle Secrets
    elif ENV_TYPE == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            s = user_secrets.get_secret(name)
            return s.strip() if s else None
        except:
            print("❌ [ERROR] Could not access Kaggle secrets. Make sure to set them up correctly.")
    return None

print(f"Detected Environment: {ENV_TYPE}")

Detected Environment: colab


In [2]:
# @title 1. Install Dependencies
# Note: You can safely ignore errors regarding 'cudf' or 'cuml' - these are unrelated RAPIDS libraries.
!pip install -q -U fastapi uvicorn pyngrok nest_asyncio requests==2.32.4 python-dotenv
!pip install -q -U transformers accelerate
!pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# Fix for bitsandbytes: install latest version for 4-bit support
!pip install -q -U bitsandbytes nvidia-cuda-runtime-cu12 nvidia-nvjitlink-cu12

In [3]:
# @title 2. Load Roleplay Specialist
import torch, os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from threading import Thread

# Kaggle Multi-GPU fix
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# --- AUTH ---
HF_TOKEN = get_secret('HF_TOKEN')
if not HF_TOKEN:
    print("⚠️ WARNING: HF_TOKEN not found. Public models only.")
# ------------

model_id = "Sao10K/L3-8B-Stheno-v3.2"

print(f"Loading {model_id}... Using Multi-GPU distribution.")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_attn_implementation="flash_attention_2", # Speed boost if supported
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="balanced", # Force split across all available GPUs
    trust_remote_code=True,
    token=HF_TOKEN
)

print("\n🚀 GPU Allocation Breakdown:")
for i in range(torch.cuda.device_count()):
    mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
    used = torch.cuda.memory_allocated(i) / 1024**3
    name = torch.cuda.get_device_name(i)
    print(f"  [GPU {i}] {name}: {used:.2f}GB / {mem:.2f}GB used")

print(f"\n✅ Model LOADED and Distributed!")

Loading Sao10K/L3-8B-Stheno-v3.2... This may take a few minutes.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


✅ Model LOADED on Tesla T4!


In [4]:
# @title 3. Start API Server & Tunnel
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import uvicorn, nest_asyncio, re, os, time, random, threading
from pyngrok import ngrok
from pydantic import BaseModel
from typing import List
from threading import Thread
from transformers import TextIteratorStreamer
import torch

NGROK_TOKEN = get_secret('NGROK_TOKEN')
if NGROK_TOKEN:
    print(f"Applying Ngrok Token (len: {len(NGROK_TOKEN)})...")
    ngrok.set_auth_token(NGROK_TOKEN)
else:
    print("❌ ERROR: NGROK_TOKEN not found!")

app = FastAPI()
nest_asyncio.apply()
gpu_lock = threading.Lock()

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    messages: List[Message]
    max_tokens: int = 1024
    temperature: float = 0.8
    n: int = 1

@app.post("/chat")
async def chat_endpoint(request: ChatRequest):
    if gpu_lock.locked():
        print("⏳ GPU Busy... Queuing request.")

    with gpu_lock:
        chat = [{"role": m.role, "content": m.content} for m in request.messages]
        history_len = len(chat)

        if request.n > 1:
            print(f"🎭 [BATCH] Generating {request.n} candidates (History: {history_len} turns)")
        else:
            print(f"💬 [STREAM] Thinking... (History: {history_len} turns)")

        model_inputs = tokenizer.apply_chat_template(
            chat,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        ).to(model.device)

        # Batch Generation (Non-Streaming)
        if request.n > 1:
            try:
                with torch.no_grad():
                    output_tokens = model.generate(
                        **model_inputs,
                        max_new_tokens=request.max_tokens,
                        temperature=request.temperature if request.temperature > 0 else 1.0,
                        do_sample=True if request.temperature > 0 else False,
                        top_p=0.9 if request.temperature > 0 else 1.0,
                        num_return_sequences=request.n,
                        pad_token_id=tokenizer.eos_token_id
                    )

                input_len = model_inputs.input_ids.shape[1]
                replies = []
                for i in range(request.n):
                    decoded = tokenizer.decode(output_tokens[i][input_len:], skip_special_tokens=True)
                    replies.append(decoded.strip())

                print(f"✅ [BATCH] Generated {len(replies)} replies successfully!")
                return {"candidates": replies}
            except Exception as e:
                print(f"❌ [BATCH] ERROR: {e}")
                return {"error": str(e)}
            finally:
                torch.cuda.empty_cache()
                print("🧹 CUDA Cache Cleared.")

        # Single Generation (Streaming)
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

        generation_kwargs = {
            **model_inputs,
            "streamer": streamer,
            "max_new_tokens": request.max_tokens,
            "temperature": request.temperature if request.temperature > 0 else 1.0,
            "do_sample": True if request.temperature > 0 else False,
            "top_p": 0.9 if request.temperature > 0 else 1.0,
        }

        thread = Thread(target=model.generate, kwargs=generation_kwargs)
        thread.start()

        def stream_generator():
            for new_text in streamer:
                yield new_text
            torch.cuda.empty_cache()
            print("✅ [STREAM] Generation complete. Cache cleared.")

        return StreamingResponse(stream_generator(), media_type="text/plain")

ngrok.kill()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

server_thread = Thread(target=run_server)
server_thread.daemon = True
server_thread.start()

time.sleep(2)

if server_thread.is_alive():
    try:
        public_url = ngrok.connect(8000).public_url
        print("="*50)
        print(f"\n🚀 BRIDGE ONLINE!\n")
        print(f"Copy this URL to your settings.json -> remote_llm_url:")
        print(f"{public_url}\n")
        print("="*50)
    except Exception as e:
        print(f"❌ NGROK ERROR: {e}")
else:
    print("❌ SERVER ERROR: Failed to start FastAPI.")

try:
    while True: time.sleep(1)
except KeyboardInterrupt:
    print("Bridge stopped.")

Applying Ngrok Token (len: 49)...

🚀 BRIDGE ONLINE!

Copy this URL to your settings.json -> remote_llm_url:
https://aerobically-meddlesome-ria.ngrok-free.dev

🎭 [BATCH] Generating 3 candidates (History: 7 turns)
❌ [BATCH] ERROR: CUDA out of memory. Tried to allocate 836.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 483.81 MiB is free. Including non-PyTorch memory, this process has 14.09 GiB memory in use. Of the allocated memory 13.10 GiB is allocated by PyTorch, and 883.96 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
🧹 CUDA Cache Cleared.


[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


💬 [STREAM] Thinking... (History: 7 turns)
💬 [STREAM] Thinking... (History: 7 turns)
💬 [STREAM] Thinking... (History: 7 turns)


[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


✅ [STREAM] Generation complete. Cache cleared.
✅ [STREAM] Generation complete. Cache cleared.
✅ [STREAM] Generation complete. Cache cleared.
🎭 [BATCH] Generating 3 candidates (History: 9 turns)
❌ [BATCH] ERROR: CUDA out of memory. Tried to allocate 978.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 253.81 MiB is free. Including non-PyTorch memory, this process has 14.31 GiB memory in use. Of the allocated memory 13.97 GiB is allocated by PyTorch, and 199.80 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
🧹 CUDA Cache Cleared.


...

💬 [STREAM] Thinking... (History: 9 turns)
💬 [STREAM] Thinking... (History: 9 turns)
💬 [STREAM] Thinking... (History: 9 turns)


[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


✅ [STREAM] Generation complete. Cache cleared.
✅ [STREAM] Generation complete. Cache cleared.
✅ [STREAM] Generation complete. Cache cleared.
🎭 [BATCH] Generating 2 candidates (History: 9 turns)
✅ [BATCH] Generated 2 replies successfully!
🧹 CUDA Cache Cleared.


[transformers] Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


💬 [STREAM] Thinking... (History: 10 turns)
✅ [STREAM] Generation complete. Cache cleared.
Bridge stopped.
